In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


# **Project Title:** Car Recommendation System Using Large Language Models and Vector Databases

**Overview:**
This project develops a personalized car recommendation system leveraging advanced AI techniques, including Large Language Models (LLMs), few-shot prompting, function calling, and vector databases. The system utilizes the Gemini API for LLM functionalities and integrates a vector database to efficiently store and retrieve detailed car information, enhancing the accuracy and relevance of recommendations.

**Key Components:**

- **Gemini API Integration:** Utilizes Gemini API for LLM capabilities, enabling natural language understanding and generation to process user queries and provide context-aware recommendations.

- **Few-Shot Prompting:** Employs few-shot prompting techniques to train the model on specific car-related tasks, improving its ability to generate accurate responses based on minimal examples.

- **Function Calling:** Incorporates function calling to execute specific actions or retrieve data dynamically, allowing the system to interact with external databases or APIs as needed.

- **Vector Database:** Stores car information in a vector database, facilitating efficient similarity searches and enabling the system to recommend cars based on semantic similarity to user preferences.

**Objective:**
To build an intelligent car recommendation system that understands user preferences expressed in natural language and provides personalized car suggestions by leveraging semantic search capabilities of vector databases.




# **installing all the required libraries**



In [ ]:


!pip install -U -q "google-genai==1.7.0"
!pip install langchain
!pip install wikipedia
!pip install selenium
!pip install chromadb
!pip install sentence_transformers


# **gemmni api key**



In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GOOGLE_API_KEY")

api=secret_value_0


# **vector database using chromadb**



In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

#load the datasets
df=pd.read_csv("https://raw.githubusercontent.com/shivanshu099/free_dataset_github/refs/heads/main/indian_car.csv")
#df.head(1)

# Choose a model from Hugging Face (e.g., 'all-mpnet-base-v2')
model = SentenceTransformer('all-mpnet-base-v2')
#client=chromadb.Client()
# **Persist the ChromaDB client**
persist_directory = "chroma_db"  # Specify a directory for persistence
client = chromadb.PersistentClient(path=persist_directory)

collection=client.create_collection("car_dataset_embeddings1")
for index ,row in df.iterrows():
  temp=' '.join(str(x) for x in row.values)
  embed=model.encode(temp).tolist()
  collection.add(
      documents=[temp],
      metadatas=[{"source": str(index)}],
      ids=[str(index)],
      embeddings=[embed]
  )





# **database searching function**



In [ ]:
from typing import List

def database_query(query:str)->str:
  query_embed=model.encode(query).tolist()
  results=collection.query(
    query_embeddings=query_embed,
    n_results=10
  )
  return results['documents'][0][0]
  




# **llm model instruction**



In [ ]:
model_instructions = """
You are Yumi, a car recommendation system. Your job is to provide personalized car recommendations based on user preferences and vehicle attributes. You have access to a wiki_search tool, which you can use to retrieve relevant and specifications  information from Wikipedia as needed.
and do not talk other than the car and you can also check for avaliable car uisng database_query that give avalible car in showroom
Solve the car recommendation task using interleaved Thought, Action, and Observation steps:

1. **Thought:** Analyze the user’s preferences and the desired vehicle attributes.
2. **Action:** Execute the following operations as required:
    - `<search>entity</search>`: Retrieve detailed information about a specific car model from a database or API.
    - `<lookup>keyword</lookup>`: Search for key features or specifications within the current context.
    - `<finish>recommendation</finish>`: Conclude by providing a personalized car recommendation based on the accumulated data.
3. **Observation:** Reflect on the results of your actions to further refine the recommendation if necessary.

Follow these steps systematically to arrive at a coherent and accurate recommendation.
"""

example1 = """User Query
I'm looking for a family-friendly SUV with good fuel efficiency and a budget around $35,000.

Thought 1
The user seeks a family-friendly SUV with good fuel efficiency within a $35,000 budget. I need to search for SUVs that meet these criteria.

Action 1
<search>family-friendly SUVs with good fuel efficiency under $35,000</search>

Observation 1
Models like the Toyota RAV4, Honda CR-V, and Ford Escape are known for their family-friendly features and fuel efficiency within the specified budget.

Thought 2
Considering the user's emphasis on fuel efficiency and family-friendliness, the Toyota RAV4 stands out as a top choice.

Action 2
<finish>Based on your preferences, the Toyota RAV4 is a highly recommended option. It offers ample space for families and excellent fuel efficiency, all within your budget.</finish>
"""

example2 = """User Query
Can you suggest a compact sedan with advanced safety features and a price below $25,000?

Thought 1
The user desires a compact sedan with advanced safety features under $25,000. I should search for sedans that offer these features within the budget.

Action 1
<search>compact sedans with advanced safety features under $25,000</search>

Observation 1
Vehicles like the Honda Civic, Toyota Corolla, and Hyundai Elantra are recognized for their safety features and affordability.

Thought 2
Among these, the Honda Civic is renowned for its comprehensive safety suite and reliability.

Action 2
<finish>The Honda Civic is an excellent choice, offering advanced safety features and a price point below $25,000.</finish>
"""




# **wikipedia function**



In [ ]:




import wikipedia

def wiki_search(query: str) -> str:
    """Searches Wikipedia for information on a given query."""
    try:
        # Fetch the page summary
        result = wikipedia.summary(query, sentences=3)  # Adjust sentences as needed
        return result
    except wikipedia.exceptions.PageError:
        return "Error: Page not found."
    except wikipedia.exceptions.DisambiguationError as e:
        # Handle disambiguation by selecting the first option (you can customize this)
        options = e.options
        if options:
            first_option = options[0]
            try:
                result = wikipedia.summary(first_option, sentences=3)
                return f"Disambiguation: Using '{first_option}'.\n{result}"
            except wikipedia.exceptions.PageError:
                return f"Error: Page not found for '{first_option}'."
        else:
            return "Error: Disambiguation failed."
    except Exception as e:
        return f"Error: {e}"






# **llm **



In [ ]:


from google import genai
from google.genai import types

#query="I'm looking for a reliable family car under ₹10 lakh with good fuel efficiency. Any suggestions? with high horse power"


db_tools=[wiki_search,database_query]

client=genai.Client(api_key=api)
short_config=types.GenerateContentConfig(
    maxOutputTokens=1000,
    stop_sequences=["\nObservations"],
    system_instruction=model_instructions+example1+example2,
    temperature=0.7,
    tools=db_tools
    )
def llm(query):
  response=client.models.generate_content(
    model="gemini-2.0-flash",
    contents=f"{query}",
    config=short_config

  )
  print(response.text)
  #speak(response.text)

while True:
  query=input("enter the query ")
  if query:
    llm(query)







